In [1]:
import math
import tqdm
import torch
import gpytorch
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import NearestNeighbors
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from deep_gp.deep_kernel_class import LargeFeatureExtractor, GPRegressionModel

%matplotlib inline
%load_ext autoreload
%autoreload 2



In [2]:

features = pd.read_csv('../features.csv')
targets = pd.read_csv('../targets.csv')
data = features.merge(targets[['study_id','patient_id','case_ISUP']], on='study_id')

# predictors and target (include patient_id if you want, as in your code)
X = data.drop(['study_id','case_ISUP'], axis=1)
y = data['case_ISUP']

In [3]:
df = X.copy()
df['label'] = y

# target per class
N = len(df)
T = N // 6
print("Target per class:", T)

# Fill minority classes using class 0
idx_class0 = df[df['label'] == 0].index
used = set()

for cls in range(1, 6):
    current = (df['label'] == cls).sum()
    needed = T - current
    print(f"Class {cls} needs {needed}")
    
    if needed <= 0:
        continue
    
    cls_idx = df[df['label'] == cls].index
    knn = NearestNeighbors(n_neighbors=1)
    knn.fit(df.loc[cls_idx, X.columns])
    
    distances, indices = knn.kneighbors(df.loc[idx_class0, X.columns])
    sorted_idx = np.argsort(distances.flatten())
    
    selected = []
    for i in sorted_idx:
        idx0 = idx_class0[i]
        if idx0 not in used:
            selected.append(idx0)
            used.add(idx0)
        if len(selected) == needed:
            break
    
    df.loc[selected, 'label'] = cls

# Reduce class 0 to target size
idx_class0 = df[df['label'] == 0].index
extra = len(idx_class0) - T
print("Class 0 extra samples:", extra)

if extra > 0:
    idx_other = df[df['label'] != 0].index
    
    knn2 = NearestNeighbors(n_neighbors=1)
    knn2.fit(df.loc[idx_other, X.columns])
    
    distances, indices = knn2.kneighbors(df.loc[idx_class0, X.columns])
    sorted_idx = np.argsort(distances.flatten())
    
    selected = idx_class0[sorted_idx[:extra]]
    nearest_classes = df.loc[idx_other].iloc[indices[sorted_idx[:extra]].flatten()]['label'].values
    
    df.loc[selected, 'label'] = nearest_classes

print("\nFinal balanced counts:")
print(df['label'].value_counts().sort_index())

X_balanced = df.drop(columns=['label'])
y_balanced = df['label']


Target per class: 171
Class 1 needs 14
Class 2 needs 17
Class 3 needs 102
Class 4 needs 144
Class 5 needs 136
Class 0 extra samples: 5

Final balanced counts:
label
0    171
1    172
2    175
3    171
4    171
5    171
Name: count, dtype: int64


In [4]:
# Convert balanced data to torch tensors
X_tensor = torch.tensor(X_balanced.values, dtype=torch.float32)
y_tensor = torch.tensor(y_balanced.values, dtype=torch.float32)

# Train/test split like the elevators example (on balanced data)
train_n = int(math.floor(0.8 * len(X_tensor)))

train_x = X_tensor[:train_n, :].contiguous()
train_y = y_tensor[:train_n].contiguous()

test_x = X_tensor[train_n:, :].contiguous()
test_y = y_tensor[train_n:].contiguous()

# Move to GPU if available
if torch.cuda.is_available():
    train_x, train_y, test_x, test_y = (
        train_x.cuda(), train_y.cuda(), test_x.cuda(), test_y.cuda()
    )

print("Train set shape:", train_x.shape, train_y.shape)
print("Test set shape:", test_x.shape, test_y.shape)


Train set shape: torch.Size([824, 109]) torch.Size([824])
Test set shape: torch.Size([207, 109]) torch.Size([207])


In [5]:
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = GPRegressionModel(train_x, train_y, likelihood, data_dim=train_x.shape[1])

if torch.cuda.is_available():
    model = model.cuda()
    likelihood = likelihood.cuda()

training_iterations = 1000
model.train()
likelihood.train()

optimizer = torch.optim.Adam([
    {'params': model.feature_extractor.parameters()},
    {'params': model.covar_module.parameters()},
    {'params': model.mean_module.parameters()},
    {'params': model.likelihood.parameters()},
], lr=0.01)

mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iterations):
    optimizer.zero_grad()
    output = model(train_x)
    loss = -mll(output, train_y)
    loss.backward()
    optimizer.step()
    if (i+1) % 10 == 0:
        print(f"Iter {i+1}/{training_iterations} - Loss: {loss.item():.3f}")


/home/chrysoula/.cache/pypoetry/virtualenvs/deep-gp-BifO3guL-py3.11/lib/python3.11/site-packages/linear_operator/utils/sparse.py:51: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  if nonzero_indices.storage():
/home/chrysoula/.cache/pypoetry/virtualenvs/deep-gp-BifO3guL-py3.11/lib/python3.11/site-packages/linear_operator/utils/sparse.py:66: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  res = cls(index_tensor, value_tensor, interp_size)
/home/chrysoula/.cache/pypoetry/virtualenvs/deep-gp-BifO3guL-py3.11/lib/python3.11/site-packages/linear_operator/

Iter 10/1000 - Loss: 2.352
Iter 20/1000 - Loss: 2.265
Iter 30/1000 - Loss: 2.206
Iter 40/1000 - Loss: 2.154
Iter 50/1000 - Loss: 2.104
Iter 60/1000 - Loss: 2.066
Iter 70/1000 - Loss: 2.036
Iter 80/1000 - Loss: 2.007
Iter 90/1000 - Loss: 1.973
Iter 100/1000 - Loss: 1.944
Iter 110/1000 - Loss: 1.928
Iter 120/1000 - Loss: 1.911
Iter 130/1000 - Loss: 1.894
Iter 140/1000 - Loss: 1.884
Iter 150/1000 - Loss: 1.880
Iter 160/1000 - Loss: 1.870
Iter 170/1000 - Loss: 1.859
Iter 180/1000 - Loss: 1.853
Iter 190/1000 - Loss: 1.851
Iter 200/1000 - Loss: 1.845
Iter 210/1000 - Loss: 1.843
Iter 220/1000 - Loss: 1.836
Iter 230/1000 - Loss: 1.840
Iter 240/1000 - Loss: 1.832
Iter 250/1000 - Loss: 1.822
Iter 260/1000 - Loss: 1.824
Iter 270/1000 - Loss: 1.823
Iter 280/1000 - Loss: 1.824
Iter 290/1000 - Loss: 1.814
Iter 300/1000 - Loss: 1.811
Iter 310/1000 - Loss: 1.809
Iter 320/1000 - Loss: 1.816
Iter 330/1000 - Loss: 1.816
Iter 340/1000 - Loss: 1.811
Iter 350/1000 - Loss: 1.814
Iter 360/1000 - Loss: 1.811
I

In [ ]:
model.eval()
likelihood.eval()

with torch.no_grad(), gpytorch.settings.use_toeplitz(False), gpytorch.settings.fast_pred_var():
    preds = likelihood(model(test_x))

f_mean = preds.mean
f_var = preds.variance

lower = f_mean - 2 * f_var.sqrt()   # the classic 68–95–99.7 rule (2 standard deviations → 95% confidence)
upper = f_mean + 2 * f_var.sqrt()

for i in range(5):
    print(f"Predicted mean: {f_mean[i].item():.2f}, "
          f"95% CI: [{lower[i].item():.2f}, {upper[i].item():.2f}], "
          f"True: {test_y[i].item()}")


Predicted mean: 3.15, 95% CI: [0.29, 6.00], True: 1.0
Predicted mean: 3.05, 95% CI: [0.20, 5.90], True: 4.0
Predicted mean: 2.83, 95% CI: [-0.02, 5.68], True: 3.0
Predicted mean: 2.72, 95% CI: [-0.13, 5.57], True: 1.0
Predicted mean: 2.71, 95% CI: [-0.14, 5.56], True: 1.0


In [ ]:
preds = likelihood(model(test_x))
f_mean = preds.mean
f_var = preds.variance


# Convert tensors to numpy
y_true = test_y.detach().cpu().numpy()
y_pred = f_mean.detach().cpu().numpy()

mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("Deep Kernel GP Performance (Balanced Data)")
print("MSE:", mse)
print("R²:", r2)



--- Deep Kernel GP Performance (Balanced Data) ---
MSE: 2.3847267627716064
R²: 0.21999967098236084


In [ ]:
f_std = f_var.sqrt()
df_unc = pd.DataFrame({
    "true": test_y.detach().cpu().numpy(),
    "std": f_std.detach().cpu().numpy()
})
print(" Predictive Uncertainty per ISUP Class")
print(df_unc.groupby("true")["std"].mean())



--- Predictive Uncertainty per ISUP Class ---
true
0.0    1.474746
1.0    1.427261
2.0    1.428310
3.0    1.425030
4.0    1.438672
5.0    1.430365
Name: std, dtype: float32
